# 04 - Empirical Netflix Privacy-Utility Study

This notebook is the cohesive data-science study for the project. It treats Narayanan and Shmatikov's Netflix de-anonymization result as the starting point, then asks a modern guardrail question:

> If a platform must release user-item preference data for research or recommender modeling, which transformations reduce identity risk while preserving downstream utility?

The study combines six pieces of evidence:

1. Dataset sparsity and heavy-tailed movie popularity.
2. IMDb-assisted probabilistic record linkage.
3. k-anonymity and sampled linkage risk under blue-team releases.
4. scikit-learn nearest-neighbor and membership-inference red-team attacks.
5. SDV synthetic data generation and train-on-synthetic/test-on-real utility.
6. PySpark ALS recommender utility and factor diagnostics for targeted anonymization.

The notebook is intentionally empirical. It produces tables and figures that can be rerun at a smaller scale for iteration or a larger scale for final reporting.

## Study Design and Threat Model

The original Netflix paper argues that sparse, high-dimensional preference records behave differently from ordinary tabular quasi-identifiers. A movie rating is rarely an identifier by itself, but a small set of ratings can become highly identifying because the item universe is large, long-tailed, and user histories are idiosyncratic.

We model three red-team capabilities:

- **Auxiliary-profile linkage:** the attacker knows a few public movie ratings for a target, such as IMDb ratings, and searches for a matching anonymous Netflix customer.
- **Nearest-neighbor linkage:** the attacker has a small external target profile and asks which anonymous profile is closest in sparse rating space.
- **Membership inference:** the attacker asks whether a target user's profile is present in a released dataset at all.

We model four blue-team defenses:

- Remove or generalize time.
- Coarsen or perturb ratings.
- Suppress low-k facts and prune rare movies.
- Replace real rows with synthetic rows and evaluate them with the same attacks.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None

from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from guardrails_sensitive_data.anonymization import (
    ReleaseData,
    add_time_and_rating_features,
    evaluate_release_linkage_risk,
    evaluate_releases,
    fact_k_anonymity,
    make_releases,
    summarize_linkage_trials,
)
from guardrails_sensitive_data.imdb import read_imdb_ratings_csv
from guardrails_sensitive_data.linkage import LinkageConfig, run_linkage_attack
from guardrails_sensitive_data.netflix_io import netflix_paths, read_movie_titles, read_netflix_ratings
from guardrails_sensitive_data.paths import DEFAULT_OUTPUT_DIR
from guardrails_sensitive_data.recommender import fit_bias_recommender, mae, rmse, train_test_split_ratings

if sns is not None:
    sns.set_theme(style="whitegrid")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = ROOT / "data" / "netflix"
REPORT_DIR = ROOT / "reports"
SEED = 333
MAX_ROWS = 300_000
PRIVACY_TRIALS = 120
PROFILE_USERS = 250
PROFILE_FACTS = 6
SYNTH_TRAIN_ROWS = 60_000
USE_CACHED_REPORTS = True

SKLEARN_AVAILABLE = importlib.util.find_spec("sklearn") is not None
SDV_AVAILABLE = importlib.util.find_spec("sdv") is not None
PYSPARK_AVAILABLE = importlib.util.find_spec("pyspark") is not None

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
print({"sklearn": SKLEARN_AVAILABLE, "sdv": SDV_AVAILABLE, "pyspark": PYSPARK_AVAILABLE})

## 1. Load a Working Netflix Sample

The Netflix Prize data is large enough that every empirical claim should state its sample size. The default here is a manageable sample for exploration. For final figures, increase `MAX_ROWS` in the setup cell.

In [ ]:
paths = netflix_paths(DATA_DIR)
ratings = read_netflix_ratings(DATA_DIR, max_rows=MAX_ROWS, include_date=True)
ratings = add_time_and_rating_features(ratings)
titles = read_movie_titles(paths.movie_titles_file)

summary = pd.DataFrame([
    {
        "ratings": len(ratings),
        "customers": ratings["customer_id"].nunique(),
        "movies": ratings["movie_id"].nunique(),
        "months": ratings["month"].nunique(),
        "mean_rating": ratings["rating"].mean(),
        "median_user_history": ratings.groupby("customer_id").size().median(),
        "median_movie_popularity": ratings.groupby("movie_id").size().median(),
    }
])
display(summary)
ratings.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ratings["rating"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Share of rows")

user_lengths = ratings.groupby("customer_id").size()
axes[1].hist(user_lengths.clip(upper=user_lengths.quantile(0.99)), bins=35, color="#F58518", alpha=0.8)
axes[1].set_title("User history lengths")
axes[1].set_xlabel("Ratings per user, clipped at p99")
axes[1].set_ylabel("Users")

movie_pop = ratings["movie_id"].value_counts().sort_values(ascending=False).reset_index(drop=True)
axes[2].plot(np.arange(1, len(movie_pop) + 1), movie_pop, color="#54A24B")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_title("Movie popularity is long-tailed")
axes[2].set_xlabel("Movie rank")
axes[2].set_ylabel("Ratings")
plt.tight_layout()

**Interpretation.** The privacy problem starts here. The long tail means many movies are rare enough to be informative. A user's profile is not just a row in a table; it is a sparse fingerprint across thousands of possible items. This is why the original paper emphasized sparsity rather than fixed quasi-identifiers.

## 2. IMDb-Assisted Probabilistic Record Linkage

The first red-team attack uses the project's existing IMDb-to-Netflix title matching and probabilistic candidate scoring. If cached reports exist, this notebook reads them for speed. Otherwise, it runs the linkage attack from the package.

In [ ]:
def find_imdb_csv() -> Path | None:
    candidates = [ROOT / "notebooks" / "imdb_data.csv", ROOT / "notebooks" / "old" / "imdb_data.csv"]
    return next((path for path in candidates if path.exists()), None)

user = "planktonrules"
cached_candidates = REPORT_DIR / f"linkage_{user}_candidates.csv"
cached_facts = REPORT_DIR / f"linkage_{user}_facts.csv"

if USE_CACHED_REPORTS and cached_candidates.exists() and cached_facts.exists():
    linkage_candidates = pd.read_csv(cached_candidates)
    linkage_facts = pd.read_csv(cached_facts)
    linkage_source = "cached reports"
else:
    imdb_csv = find_imdb_csv()
    if imdb_csv is None:
        linkage_candidates = pd.DataFrame()
        linkage_facts = pd.DataFrame()
        linkage_source = "skipped: no IMDb CSV found"
    else:
        imdb_ratings = read_imdb_ratings_csv(imdb_csv)
        _matched, linkage_facts, linkage_candidates = run_linkage_attack(
            imdb_ratings,
            titles,
            DATA_DIR,
            user=user,
            linkage_config=LinkageConfig(min_matches=2, top_n=50),
        )
        linkage_source = f"computed from {imdb_csv}"

print(linkage_source)
display(linkage_candidates.head(12))

In [ ]:
if not linkage_candidates.empty:
    plot_frame = linkage_candidates.head(20).copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(plot_frame["rank"].astype(str), plot_frame["log_score"], color="#4C78A8")
    ax.invert_yaxis()
    ax.set_title(f"Top probabilistic linkage candidates for {user}")
    ax.set_xlabel("Log score, higher is more compatible")
    ax.set_ylabel("Candidate rank")
    plt.tight_layout()

    top = linkage_candidates.iloc[0]
    display(Markdown(
        f"**Linkage takeaway.** The top candidate matched {int(top['matched_facts'])} of "
        f"{int(top['total_facts'])} facts with profile RMSE {top['rmse_to_imdb_profile']:.2f}. "
        "A small auxiliary profile can collapse a large anonymous search space into a short ranked list."
    ))
else:
    display(Markdown("No linkage candidates were available. Add `notebooks/imdb_data.csv` or run `python main.py linkage-attack --user planktonrules`."))

## 3. Blue-Team Release Variants

Now we evaluate the release transformations that the package already implements: removing dates, generalizing dates, coarsening ratings, adding noise, pruning rare movies, suppressing low-k facts, and movie-only release.

The two key privacy summaries are:

- **Fact k-anonymity:** how many users share each released fact tuple.
- **Sampled linkage risk:** if an attacker knows `n` facts about a target, how many candidate users remain?

In [ ]:
releases = make_releases(ratings, seed=SEED, rare_movie_min_users=100, k_suppression=5)

k_rows = []
trial_frames = []
for release in releases:
    k_rows.append({
        "release_name": release.name,
        "knowledge_cols": " + ".join(release.knowledge_cols),
        "rows_released": len(release.data),
        "rating_col": release.rating_col,
        **fact_k_anonymity(release.data, release.knowledge_cols),
    })
    trial_frames.append(evaluate_release_linkage_risk(release, n_known_values=(1, 2, 3), trials=PRIVACY_TRIALS, seed=SEED))

k_summary = pd.DataFrame(k_rows)
privacy_trials = pd.concat(trial_frames, ignore_index=True)
privacy_summary = summarize_linkage_trials(privacy_trials)
display(k_summary.sort_values("min_k").head(10))
display(privacy_summary.head(12))

In [ ]:
plot_privacy = privacy_summary[privacy_summary["n_known"].isin([1, 2, 3])].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if sns is not None:
    sns.barplot(data=plot_privacy, x="n_known", y="median_candidate_size", hue="release_name", ax=axes[0])
    sns.barplot(data=plot_privacy, x="n_known", y="pct_unique", hue="release_name", ax=axes[1])
else:
    plot_privacy.pivot(index="n_known", columns="release_name", values="median_candidate_size").plot(kind="bar", ax=axes[0])
    plot_privacy.pivot(index="n_known", columns="release_name", values="pct_unique").plot(kind="bar", ax=axes[1])

axes[0].set_title("Candidate set size after sampled linkage")
axes[0].set_ylabel("Median candidate count, higher is safer")
axes[1].set_title("Unique re-identification rate")
axes[1].set_ylabel("Percent unique, lower is safer")
for ax in axes:
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()

**Interpretation.** A privacy-preserving release should push median candidate sets up and unique identification down. Movie-only releases usually look safest but lose rating labels. Date removal and rating coarsening are more useful compromises because they preserve some downstream learning signal.

## 4. scikit-learn Nearest-Neighbor and Membership-Inference Attacks

The original paper's scoring attack can be viewed as a sparse nearest-neighbor problem. Here we use scikit-learn's `NearestNeighbors` on a sparse user-by-movie rating matrix, then use `LogisticRegression` to classify whether a target profile appears in the release.

In [ ]:
if not SKLEARN_AVAILABLE:
    display(Markdown("scikit-learn is not installed. Run `python -m pip install -e \".[ml]\"` to execute this section."))
    nn_summary = pd.DataFrame()
else:
    from scipy.sparse import csr_matrix
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.neighbors import NearestNeighbors

    def split_by_user(frame: pd.DataFrame, holdout_fraction: float = 0.25) -> tuple[pd.DataFrame, pd.DataFrame]:
        rng = np.random.default_rng(SEED)
        users = frame["customer_id"].drop_duplicates().to_numpy()
        holdout_users = set(rng.choice(users, size=max(1, int(holdout_fraction * len(users))), replace=False).astype(int))
        return frame[~frame["customer_id"].isin(holdout_users)].copy(), frame[frame["customer_id"].isin(holdout_users)].copy()

    def fit_user_matrix(frame: pd.DataFrame, rating_col: str = "rating"):
        small = frame[["customer_id", "movie_id", rating_col]].dropna().rename(columns={rating_col: "rating"})
        user_index = pd.Index(small["customer_id"].drop_duplicates())
        movie_index = pd.Index(small["movie_id"].drop_duplicates())
        rows = user_index.get_indexer(small["customer_id"])
        cols = movie_index.get_indexer(small["movie_id"])
        values = small["rating"].to_numpy(dtype=np.float32) - 3.0
        matrix = csr_matrix((values, (rows, cols)), shape=(len(user_index), len(movie_index)))
        return user_index, movie_index, matrix

    def sample_profiles(frame: pd.DataFrame, n_users: int, n_facts: int, seed: int) -> dict[int, pd.DataFrame]:
        rng = np.random.default_rng(seed)
        eligible = frame["customer_id"].value_counts().loc[lambda s: s >= n_facts].index.to_numpy()
        users = rng.choice(eligible, size=min(n_users, len(eligible)), replace=False)
        return {
            int(user): frame[frame["customer_id"] == user].sample(n_facts, random_state=int(rng.integers(2**31)))[["movie_id", "rating"]]
            for user in users
        }

    def profile_vector(profile: pd.DataFrame, movie_index: pd.Index) -> csr_matrix:
        cols = movie_index.get_indexer(profile["movie_id"])
        keep = cols >= 0
        values = profile.loc[keep, "rating"].to_numpy(dtype=np.float32) - 3.0
        return csr_matrix((values, ([0] * int(keep.sum()), cols[keep])), shape=(1, len(movie_index)))

    def audit_release_with_sklearn(release_frame: pd.DataFrame, rating_col: str, positive_profiles, negative_profiles) -> dict[str, float]:
        user_index, movie_index, matrix = fit_user_matrix(release_frame, rating_col=rating_col)
        n_neighbors = min(10, max(1, matrix.shape[0]))
        nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=n_neighbors).fit(matrix)

        nearest_rows = []
        for target_user, profile in positive_profiles.items():
            distances, positions = nn.kneighbors(profile_vector(profile, movie_index), n_neighbors=n_neighbors)
            candidates = user_index.take(positions[0]).astype(int).tolist()
            nearest_rows.append({"top1_hit": candidates[0] == target_user, "topk_hit": target_user in candidates, "best_similarity": 1 - float(distances[0][0])})
        nearest = pd.DataFrame(nearest_rows)

        def features(profiles):
            rows = []
            for _target_user, profile in profiles.items():
                distances, _positions = nn.kneighbors(profile_vector(profile, movie_index), n_neighbors=min(5, n_neighbors))
                sims = 1 - distances[0]
                second = sims[1] if len(sims) > 1 else sims[0]
                rows.append({"best_similarity": float(sims[0]), "similarity_gap": float(sims[0] - second), "mean_top_similarity": float(sims.mean())})
            return pd.DataFrame(rows)

        X = pd.concat([features(positive_profiles), features(negative_profiles)], ignore_index=True)
        y = np.r_[np.ones(len(positive_profiles)), np.zeros(len(negative_profiles))]
        if len(np.unique(y)) < 2 or len(X) < 10:
            auc = np.nan
            acc = np.nan
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
            clf = LogisticRegression(max_iter=1_000).fit(X_train, y_train)
            scores = clf.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, scores)
            acc = accuracy_score(y_test, scores >= 0.5)
        return {"top1_hit_rate": nearest["top1_hit"].mean(), "top10_hit_rate": nearest["topk_hit"].mean(), "membership_auc": auc, "membership_accuracy": acc}

    release_users, holdout_users = split_by_user(ratings)
    positive_profiles = sample_profiles(release_users, PROFILE_USERS, PROFILE_FACTS, SEED)
    negative_profiles = sample_profiles(holdout_users, PROFILE_USERS, PROFILE_FACTS, SEED + 1)

    nn_rows = []
    for release in make_releases(release_users, seed=SEED, rare_movie_min_users=100, k_suppression=5):
        if release.rating_col is None:
            continue
        metrics = audit_release_with_sklearn(release.data, release.rating_col, positive_profiles, negative_profiles)
        nn_rows.append({"release_name": release.name, **metrics})
    nn_summary = pd.DataFrame(nn_rows).sort_values("membership_auc", ascending=False)
    display(nn_summary)

In [ ]:
if SKLEARN_AVAILABLE and not nn_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    nn_summary.sort_values("membership_auc").plot.barh(x="release_name", y="membership_auc", ax=axes[0], color="#E45756", legend=False)
    axes[0].set_title("Membership inference AUC by release")
    axes[0].set_xlabel("AUC, lower is safer")
    nn_summary.sort_values("top10_hit_rate").plot.barh(x="release_name", y="top10_hit_rate", ax=axes[1], color="#72B7B2", legend=False)
    axes[1].set_title("Nearest-neighbor top-10 hit rate")
    axes[1].set_xlabel("Hit rate, lower is safer")
    plt.tight_layout()

**Interpretation.** Nearest-neighbor attacks are a useful complement to k-anonymity. k-anonymity asks how many users share exact released facts; nearest-neighbor attacks ask whether a profile is still unusually close after noise, coarsening, or pruning. Membership AUC is especially useful for synthetic releases because it tests whether generated rows reveal training membership.

## 5. Synthetic Data Generation with SDV

Synthetic data is attractive because it can preserve aggregate distributions without releasing actual customer ids. But it is not automatically private, and it may lose recommender utility. We use SDV as a plug-and-play baseline and evaluate it with the same dashboard.

In [ ]:
if not SDV_AVAILABLE:
    display(Markdown("SDV is not installed. Run `python -m pip install -e \".[ml]\"` to execute this section."))
    synthetic = pd.DataFrame()
    synthetic_report = pd.DataFrame()
    synthetic_utility = pd.DataFrame()
else:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import GaussianCopulaSynthesizer

    synth_train = ratings[["movie_id", "rating", "month"]].sample(min(SYNTH_TRAIN_ROWS, len(ratings)), random_state=SEED).reset_index(drop=True)
    synth_train.insert(0, "row_id", np.arange(len(synth_train)))
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(synth_train)
    metadata.set_primary_key("row_id")
    for col in ["movie_id", "rating", "month"]:
        metadata.update_column(col, sdtype="categorical")

    synthesizer = GaussianCopulaSynthesizer(metadata, enforce_rounding=True)
    synthesizer.fit(synth_train)
    raw_synth = synthesizer.sample(num_rows=len(synth_train))

    def postprocess_synthetic(frame: pd.DataFrame, real: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        out["movie_id"] = pd.to_numeric(out["movie_id"], errors="coerce").round()
        out["rating"] = pd.to_numeric(out["rating"], errors="coerce").round().clip(1, 5)
        out = out.dropna(subset=["movie_id", "rating", "month"])
        valid_movies = set(real["movie_id"].astype(int).unique())
        out["movie_id"] = out["movie_id"].astype(int)
        out = out[out["movie_id"].isin(valid_movies)].copy()
        out["rating"] = out["rating"].astype("int8")
        rng = np.random.default_rng(SEED)
        lengths = real.groupby("customer_id").size().clip(upper=150).to_numpy()
        users, cursor, user_id = [], 0, 900_000_000
        while cursor < len(out):
            history_len = int(rng.choice(lengths))
            n = min(history_len, len(out) - cursor)
            users.extend([user_id] * n)
            cursor += n
            user_id += 1
        out = out.iloc[: len(users)].copy()
        out["customer_id"] = users
        out["date"] = out["month"].astype(str).where(out["month"].astype(str).eq("unknown"), out["month"].astype(str) + "-15")
        return add_time_and_rating_features(out[["movie_id", "customer_id", "rating", "date"]])

    synthetic = postprocess_synthetic(raw_synth, ratings)

    def js_divergence(left: pd.Series, right: pd.Series) -> float:
        levels = left.index.union(right.index)
        p = left.reindex(levels, fill_value=0).to_numpy(dtype=float)
        q = right.reindex(levels, fill_value=0).to_numpy(dtype=float)
        p = p / p.sum(); q = q / q.sum(); m = 0.5 * (p + q); eps = 1e-12
        return float(0.5 * np.sum(p * np.log2((p + eps) / (m + eps))) + 0.5 * np.sum(q * np.log2((q + eps) / (m + eps))))

    synthetic_report = pd.DataFrame([
        {"feature": col, "js_divergence": js_divergence(ratings[col].value_counts(normalize=True, dropna=False), synthetic[col].value_counts(normalize=True, dropna=False))}
        for col in ["movie_id", "rating", "month"]
    ])

    real_train, real_test = train_test_split_ratings(ratings, test_fraction=0.2, seed=SEED)
    actual = real_test["rating"].to_numpy(dtype=np.float32)
    real_model = fit_bias_recommender(real_train, epochs=6)
    synth_model = fit_bias_recommender(synthetic[["movie_id", "customer_id", "rating", "date"]], epochs=6)
    synthetic_utility = pd.DataFrame([
        {"training_data": "real_sample", "rmse": rmse(actual, real_model.predict(real_test)), "mae": mae(actual, real_model.predict(real_test))},
        {"training_data": "sdv_gaussian_copula", "rmse": rmse(actual, synth_model.predict(real_test)), "mae": mae(actual, synth_model.predict(real_test))},
    ])

    display(synthetic_report)
    display(synthetic_utility)

In [ ]:
if SDV_AVAILABLE and not synthetic.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ratings["rating"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.7, label="real")
    synthetic["rating"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.5, label="synthetic")
    axes[0].set_title("Rating distribution")
    axes[0].legend()

    ratings["month"].value_counts(normalize=True).sort_index().tail(36).plot(ax=axes[1], label="real")
    synthetic["month"].value_counts(normalize=True).sort_index().tail(36).plot(ax=axes[1], label="synthetic")
    axes[1].set_title("Recent month distribution")
    axes[1].tick_params(axis="x", rotation=90)
    axes[1].legend()

    synthetic_report.plot.bar(x="feature", y="js_divergence", ax=axes[2], color="#B279A2", legend=False)
    axes[2].set_title("Synthetic fidelity gap")
    axes[2].set_ylabel("Jensen-Shannon divergence")
    plt.tight_layout()

**Interpretation.** Synthetic data should be judged on three axes at once: distributional fidelity, downstream utility, and attack resistance. A generator that matches rating marginals but fails on movie co-occurrence can look good in simple plots while still being weak for recommender training.

## 6. Downstream Utility: Bias Baseline and PySpark ALS

The project already has a bias-based recommender baseline. This section computes that baseline for every release, then optionally fits Spark ALS as a stronger collaborative-filtering task. ALS is especially useful because its item factors can identify movies with high latent leverage.

In [ ]:
train, test = train_test_split_ratings(ratings, test_fraction=0.2, seed=SEED)
actual = test["rating"].to_numpy(dtype=np.float32)

utility_rows = []
for release in make_releases(train, seed=SEED, rare_movie_min_users=100, k_suppression=5):
    if release.rating_col is None:
        utility_rows.append({"release_name": release.name, "model": "bias", "rmse": np.nan, "mae": np.nan, "status": "skipped_no_rating_labels"})
        continue
    model = fit_bias_recommender(release.data.dropna(subset=[release.rating_col]), rating_col=release.rating_col, epochs=6)
    pred = model.predict(test)
    utility_rows.append({"release_name": release.name, "model": "bias", "rmse": rmse(actual, pred), "mae": mae(actual, pred), "status": "ok"})

utility_summary = pd.DataFrame(utility_rows).sort_values("rmse", na_position="last")
display(utility_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
utility_summary[utility_summary["status"] == "ok"].sort_values("rmse").plot.barh(x="release_name", y="rmse", ax=ax, color="#F58518", legend=False)
ax.set_title("Downstream utility by release: bias recommender")
ax.set_xlabel("RMSE on true held-out ratings, lower is better")
plt.tight_layout()

In [ ]:
if not PYSPARK_AVAILABLE:
    display(Markdown("PySpark is not installed. Run `python -m pip install -e \".[ml]\"` and make sure Java is on `PATH` to execute Spark ALS."))
    spark_als_summary = pd.DataFrame()
    factor_diagnostics = pd.DataFrame()
else:
    from pyspark.ml.evaluation import RegressionEvaluator
    from pyspark.ml.recommendation import ALS
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("netflix-privacy-study")
        .master("local[*]")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")

    train_sdf = spark.createDataFrame(train[["customer_id", "movie_id", "rating"]].astype({"customer_id": "int32", "movie_id": "int32", "rating": "float32"}))
    test_sdf = spark.createDataFrame(test[["customer_id", "movie_id", "rating"]].astype({"customer_id": "int32", "movie_id": "int32", "rating": "float32"}))
    als = ALS(userCol="customer_id", itemCol="movie_id", ratingCol="rating", rank=32, maxIter=10, regParam=0.08, seed=SEED, coldStartStrategy="drop")
    als_model = als.fit(train_sdf)
    spark_predictions = als_model.transform(test_sdf)
    evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
    spark_als_summary = pd.DataFrame([{"release_name": "original_train", "model": "spark_als", "rmse": evaluator.evaluate(spark_predictions)}])

    factors = als_model.itemFactors.toPandas().rename(columns={"id": "movie_id"})
    factors["factor_norm"] = factors["features"].apply(lambda xs: float(np.linalg.norm(xs)))
    counts = train["movie_id"].value_counts().rename("rating_count")
    factor_diagnostics = factors.merge(counts, left_on="movie_id", right_index=True)
    factor_diagnostics["latent_leverage"] = factor_diagnostics["factor_norm"] / np.sqrt(factor_diagnostics["rating_count"].clip(lower=1))
    factor_diagnostics = factor_diagnostics.merge(titles, on="movie_id", how="left").sort_values("latent_leverage", ascending=False)
    display(spark_als_summary)
    display(factor_diagnostics.head(12)[["movie_id", "title", "rating_count", "factor_norm", "latent_leverage"]])

In [ ]:
if PYSPARK_AVAILABLE and not factor_diagnostics.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(factor_diagnostics["rating_count"], factor_diagnostics["latent_leverage"], alpha=0.6, color="#4C78A8")
    ax.set_xscale("log")
    ax.set_title("Spark ALS factor leverage identifies high-risk movies")
    ax.set_xlabel("Movie popularity")
    ax.set_ylabel("Factor norm / sqrt(popularity)")
    plt.tight_layout()

**Interpretation.** Utility is not just a scoreboard; it can inform anonymization. High-leverage items are places where exact ratings are most informative for personalization and often most revealing for identity. That suggests targeted noise/pruning rather than uniform noise everywhere.

## 7. Global Privacy-Utility Dashboard

The final scorecard combines privacy and utility views. A good release sits toward lower RMSE, lower membership AUC, lower unique-identification rate, and higher candidate-set size. There is no single perfect metric, so the plot is meant to support discussion rather than hide tradeoffs in one number.

In [ ]:
scorecard = utility_summary[utility_summary["status"] == "ok"][["release_name", "rmse", "mae"]].copy()
privacy_n3 = privacy_summary[privacy_summary["n_known"] == 3][["release_name", "median_candidate_size", "pct_unique", "pct_10_or_less"]]
scorecard = scorecard.merge(privacy_n3, on="release_name", how="left")
if SKLEARN_AVAILABLE and not nn_summary.empty:
    scorecard = scorecard.merge(nn_summary[["release_name", "membership_auc", "top10_hit_rate"]], on="release_name", how="left")
else:
    scorecard["membership_auc"] = np.nan
    scorecard["top10_hit_rate"] = np.nan
scorecard = scorecard.sort_values(["rmse", "pct_unique"], na_position="last")
display(scorecard)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_df = scorecard.dropna(subset=["rmse", "pct_unique"]).copy()
if not plot_df.empty:
    sizes = 60 + 4 * plot_df["median_candidate_size"].fillna(1).clip(upper=200)
    scatter = ax.scatter(plot_df["rmse"], plot_df["pct_unique"], s=sizes, c=plot_df["membership_auc"].fillna(0.5), cmap="viridis", alpha=0.8)
    for row in plot_df.itertuples(index=False):
        ax.annotate(row.release_name.replace("_", "\n"), (row.rmse, row.pct_unique), fontsize=8, alpha=0.85)
    ax.set_title("Privacy-utility frontier across releases")
    ax.set_xlabel("Utility loss: RMSE, lower is better")
    ax.set_ylabel("Privacy risk: unique linkage %, lower is better")
    fig.colorbar(scatter, ax=ax, label="Membership AUC")
plt.tight_layout()

## Main Takeaways

1. **Sparsity is the core privacy risk.** The movie universe is long-tailed, and rare combinations of ratings behave like fingerprints even without names or explicit identifiers.
2. **Exact dates are high-risk.** Removing or generalizing time usually improves privacy quickly because it weakens the attacker's matching facts.
3. **Rating coarsening/noise has a real utility tradeoff.** It can reduce linkage precision, but utility should always be checked against a downstream recommender rather than assumed.
4. **Nearest-neighbor and membership attacks catch risks that k-anonymity misses.** They operate in similarity space and are especially relevant for synthetic data or noisy releases.
5. **Synthetic data is not a magic privacy blanket.** SDV-style models are useful baselines, but the release still needs attack evaluation and train-on-synthetic/test-on-real utility checks.
6. **ALS factors can guide smarter defenses.** Instead of adding isotropic noise everywhere, target high-leverage or rare latent regions where facts are most identifying.

Overall, this project reframes the 2008 Netflix lesson as a guardrail workflow: release design should be empirical, adversarial, and utility-aware.

## References

- Narayanan, A. and Shmatikov, V. **Robust De-anonymization of Large Sparse Datasets**. IEEE Symposium on Security and Privacy, 2008. https://arxiv.org/abs/cs/0610105
- Sweeney, L. **k-anonymity: a model for protecting privacy**. International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems, 2002. https://doi.org/10.1142/S0218488502001648
- Shokri, R. et al. **Membership Inference Attacks Against Machine Learning Models**. IEEE Symposium on Security and Privacy, 2017. https://arxiv.org/abs/1610.05820
- Pedregosa, F. et al. **Scikit-learn: Machine Learning in Python**. JMLR, 2011. https://arxiv.org/abs/1201.0490
- Meng, X. et al. **MLlib: Machine Learning in Apache Spark**. JMLR, 2016. https://arxiv.org/abs/1505.06807
- Koren, Y., Bell, R., and Volinsky, C. **Matrix Factorization Techniques for Recommender Systems**. Computer, 2009. https://doi.org/10.1109/MC.2009.263
- Xu, L. et al. **Modeling Tabular data using Conditional GAN**. NeurIPS, 2019. https://arxiv.org/abs/1907.00503
- McKenna, R., Miklau, G., and Sheldon, D. **Winning the NIST Contest: A scalable and general approach to differentially private synthetic data**. 2021. https://arxiv.org/abs/2108.04978
- scikit-learn `NearestNeighbors` docs: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html
- scikit-learn `LogisticRegression` docs: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- Spark collaborative filtering docs: https://spark.apache.org/docs/latest/ml-collaborative-filtering.html
- SDV synthesizer docs: https://docs.sdv.dev/sdv/single-table-data/modeling/synthesizers